# Подход 2. Расширенный датасет всех объектов недвижимости

Этот notebook содержит полный текст SQL 51 и запускает его напрямую. Внешний SQL-файл для работы notebook не нужен.

В результат попадают все неудалённые объекты `nedv_ul_and_ip`. Если подтверждённая связь с договором найдена, договорные поля заполняются. Если связь не найдена, объект остаётся в результате, а договорные поля остаются пустыми.

Преимущество подхода — значительно больше объектов. Недостаток — у несвязанных объектов нельзя использовать договорную историю, страхователя и другие договорные признаки.

## 1. Библиотеки

Следующую ячейку достаточно выполнить один раз в используемом окружении.

In [ ]:
%pip install pandas sqlalchemy "psycopg[binary]"

In [ ]:
import getpass
import subprocess
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import URL, create_engine, text

pd.set_option('display.max_columns', 100)

## 2. Путь к проекту

In [ ]:
def find_project_root(start):
    start = Path(start).resolve()
    for folder in (start, *start.parents):
        if (folder / 'docs' / 'PROJECT_CONTEXT.md').is_file():
            return folder
    raise FileNotFoundError('Не найден корень проекта')

PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
CHECK_SCRIPT = (
    PROJECT_ROOT
    / 'МАТЕРИАЛЫ_ПРОЕКТА'
    / '05_скрипты'
    / '04_проверка_датасета'
    / 'проверить_датасет_51.py'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Корень проекта:', PROJECT_ROOT)

## 3. Подключение к Сфере

Возьми сервер, порт, базу и логин из свойств подключения DBeaver. Пароль вводится скрыто и в notebook не сохраняется.

In [ ]:
SPHERE_HOST = ''       # сервер из DBeaver
SPHERE_PORT = 5432     # порт из DBeaver
SPHERE_DATABASE = ''   # база данных из DBeaver
SPHERE_USER = ''       # логин из DBeaver

if not all([SPHERE_HOST, SPHERE_DATABASE, SPHERE_USER]):
    raise ValueError('Заполни SPHERE_HOST, SPHERE_DATABASE и SPHERE_USER')

password = getpass.getpass('Пароль от Сферы: ')
connection_url = URL.create(
    drivername='postgresql+psycopg',
    username=SPHERE_USER,
    password=password,
    host=SPHERE_HOST,
    port=SPHERE_PORT,
    database=SPHERE_DATABASE,
)
engine = create_engine(connection_url, pool_pre_ping=True)

In [ ]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

## 4. SQL 51 и его запуск

Этот запрос читает все неудалённые объекты недвижимости и поэтому может работать дольше строгой выборки.

In [ ]:
sql = "/*\nзапускать в сфере\n\nзапрос собирает все неудаленные объекты недвижимости\nесли объект связан с подходящим договором данные договора заполняются\nесли связь не найдена объект остается в результате с пустыми полями договора\n\nодна строка для связанного объекта означает объект в одном договоре\nодна строка для несвязанного объекта означает его последнюю версию характеристик\n*/\n\nwith task_candidates as (\n    /* отбираем подходящие задачи оформления */\n    select\n        t.id as task_id,\n        r.id as request_id,\n        c.id as contract_id,\n        row_number() over (\n            partition by c.id\n            order by\n                coalesce(\n                    t.d_conclusion_ins_contract::timestamp with time zone,\n                    t.d_create,\n                    t.d_change\n                ) desc nulls last,\n                t.d_create desc nulls last,\n                t.d_change desc nulls last,\n                t.id desc\n        ) as task_number\n    from bps_request_ins_task t\n    join bps_request_ins r\n        on r.id = t.request_ins_id\n    join bps_contract c\n        on c.id = r.contract_id\n    where t.task_type = 'draft_contract'\n      and t.status = 'operational_archive'\n      and (\n          t.ins_document_type = 'new_ins_contract'\n          or t.ins_document_type = 'ins_contract_prolong'\n          or t.ins_document_type is null\n      )\n      and t.ins_refuse is not true\n      and t.d_delete is null\n      and r.d_delete is null\n      and c.d_delete is null\n),\n\nselected_tasks as (\n    /* оставляем последнюю подходящую задачу каждого договора */\n    select\n        task_id,\n        request_id,\n        contract_id\n    from task_candidates\n    where task_number = 1\n),\n\nlinked_object_candidates as (\n    /* находим недвижимость в выбранных задачах */\n    select\n        ch.insurance_object_id as object_id,\n        ch.id as characteristics_id,\n        link.id as task_object_link_id,\n        selected.task_id,\n        selected.request_id,\n        selected.contract_id,\n        row_number() over (\n            partition by selected.task_id, ch.insurance_object_id\n            order by\n                link.d_change desc nulls last,\n                link.d_create desc nulls last,\n                ch.version_start_date desc nulls last,\n                ch.version_number desc nulls last,\n                link.id desc\n        ) as link_number\n    from selected_tasks selected\n    join bps_request_ins_task_insurance_object link\n        on link.parent_id = selected.task_id\n    join base_insurance_object_characteristics ch\n        on ch.id = link.characteristics_id\n    join base_insurance_object obj\n        on obj.id = ch.insurance_object_id\n    where obj.elementary_obj_type = 'nedv_ul_and_ip'\n      and obj.d_delete is null\n),\n\nselected_links as (\n    /* убираем повторные связи одного объекта с одной задачей */\n    select\n        object_id,\n        characteristics_id,\n        task_object_link_id,\n        task_id,\n        request_id,\n        contract_id\n    from linked_object_candidates\n    where link_number = 1\n),\n\nobject_versions as (\n    /* нумеруем версии характеристик каждого объекта */\n    select\n        obj.id as object_id,\n        ch.id as characteristics_id,\n        row_number() over (\n            partition by obj.id\n            order by\n                ch.version_is_active desc nulls last,\n                ch.version_number desc nulls last,\n                ch.version_start_date desc nulls last,\n                ch.id desc nulls last\n        ) as version_number\n    from base_insurance_object obj\n    left join base_insurance_object_characteristics ch\n        on ch.insurance_object_id = obj.id\n    where obj.elementary_obj_type = 'nedv_ul_and_ip'\n      and obj.d_delete is null\n),\n\ndataset_keys as (\n    /* сохраняем все найденные связи с договорами */\n    select\n        linked.object_id,\n        linked.characteristics_id,\n        linked.task_object_link_id,\n        linked.task_id,\n        linked.request_id,\n        linked.contract_id,\n        'linked'::text as row_source\n    from selected_links linked\n\n    union all\n\n    /* добавляем объекты для которых подходящий договор не найден */\n    select\n        version.object_id,\n        version.characteristics_id,\n        null::integer as task_object_link_id,\n        null::integer as task_id,\n        null::integer as request_id,\n        null::integer as contract_id,\n        'not_linked'::text as row_source\n    from object_versions version\n    where version.version_number = 1\n      and not exists (\n          select 1\n          from selected_links linked\n          where linked.object_id = version.object_id\n      )\n),\n\nobject_link_profile as (\n    /* считаем со сколькими договорами связан объект */\n    select\n        object_id,\n        count(distinct contract_id) as contract_count\n    from selected_links\n    group by object_id\n),\n\nselected_characteristics as (\n    /* ограничиваем расчет условий версиями из итоговой выборки */\n    select distinct characteristics_id\n    from dataset_keys\n    where characteristics_id is not null\n),\n\ncondition_summary as (\n    /* сворачиваем варианты условий в одну строку */\n    select\n        cond.characteristics_id,\n        count(*) as condition_count,\n        count(cond.insured_sum) as filled_insured_sum_count,\n        count(distinct cond.insured_sum) filter (\n            where cond.insured_sum is not null\n        ) as distinct_insured_sum_count,\n        min(cond.insured_sum) as minimum_insured_sum,\n        max(cond.insured_sum) as maximum_insured_sum,\n        count(distinct cond.insured_sum_currency) filter (\n            where cond.insured_sum_currency is not null\n        ) as currency_count,\n        string_agg(\n            distinct cond.insured_sum_currency,\n            ', '\n            order by cond.insured_sum_currency\n        ) filter (\n            where cond.insured_sum_currency is not null\n        ) as insured_sum_currency,\n        array_agg(\n            distinct cond.terms_option_number\n            order by cond.terms_option_number\n        ) filter (\n            where cond.terms_option_number is not null\n        ) as terms_option_numbers,\n        min(cond.per_occurance_limit) as minimum_per_occurrence_limit,\n        max(cond.per_occurance_limit) as maximum_per_occurrence_limit,\n        case\n            when count(distinct cond.insured_sum) filter (\n                where cond.insured_sum is not null\n            ) = 1\n             and count(distinct cond.insured_sum_currency) filter (\n                where cond.insured_sum_currency is not null\n            ) <= 1\n            then max(cond.insured_sum)\n        end as insured_sum\n    from base_insurance_object_conditions cond\n    join selected_characteristics selected\n        on selected.characteristics_id = cond.characteristics_id\n    group by cond.characteristics_id\n)\n\nselect\n    /* качество строки */\n    keys.row_source,\n    case\n        when coalesce(profile.contract_count, 0) = 0 then 'not_linked'\n        when profile.contract_count = 1 then 'linked'\n        else 'multiple_contracts'\n    end as contract_link_status,\n    coalesce(profile.contract_count, 0) as contract_count,\n    (keys.contract_id is not null) as has_contract,\n    (address.id is not null) as has_address,\n    (conditions.insured_sum is not null) as has_target,\n    case\n        when conditions.condition_count is null then 'no_conditions'\n        when conditions.filled_insured_sum_count = 0 then 'target_is_empty'\n        when conditions.distinct_insured_sum_count > 1 then 'several_target_values'\n        when conditions.currency_count > 1 then 'several_currencies'\n        when conditions.insured_sum <= 0 then 'target_is_not_positive'\n        else 'target_is_usable'\n    end as target_status,\n\n    /* идентификаторы */\n    keys.object_id,\n    keys.characteristics_id,\n    keys.task_object_link_id,\n    keys.task_id,\n    keys.request_id,\n    keys.contract_id,\n    obj.geo_address_id,\n    contract.contractor_id,\n    request.corporate_crm_id,\n\n    /* целевая страховая сумма */\n    conditions.insured_sum,\n    conditions.insured_sum_currency,\n    conditions.condition_count,\n    conditions.filled_insured_sum_count,\n    conditions.distinct_insured_sum_count,\n    conditions.minimum_insured_sum,\n    conditions.maximum_insured_sum,\n    conditions.currency_count,\n    conditions.terms_option_numbers,\n\n    /* контрольные суммы */\n    task_link.insured_sum as task_object_insured_sum,\n    task_link.insured_sum_currency as task_object_insured_sum_currency,\n    task.total_ins_contract_amount as contract_insured_sum,\n    task.curr_ins_contract_amount as contract_insured_sum_currency,\n    task.total_ins_contract_premium as contract_premium,\n    ch.insurance_value,\n    ch.insurance_value_currency,\n    ch.insurance_value_basis,\n    ch.pledged_value,\n    conditions.minimum_per_occurrence_limit,\n    conditions.maximum_per_occurrence_limit,\n\n    /* объект */\n    obj.obj_type as object_type,\n    obj.elementary_obj_type,\n    obj.obj_name as object_name,\n    obj.description as object_description,\n    obj.original_address,\n    obj.active as object_is_active,\n    obj.d_create as object_create_date,\n    obj.d_change as object_change_date,\n\n    /* характеристики объекта */\n    ch.version_number as characteristics_version_number,\n    ch.version_start_date as characteristics_version_start_date,\n    ch.version_end_date as characteristics_version_end_date,\n    ch.version_is_active as characteristics_version_is_active,\n    ch.ownership_type,\n    ch.is_pledged,\n    ch.is_leased,\n    ch.insured_components,\n    ch.activity_types,\n    ch.risk_natures,\n    ch.insurance_territory,\n    ch.has_losses,\n    ch.insurance_object_loss_history,\n    ch.characteristics ->> 'total_area_sq_m' as total_area,\n    ch.characteristics ->> 'occupied_area_sq_m' as occupied_area,\n    ch.characteristics ->> 'construction_year' as construction_year,\n    ch.characteristics ->> 'last_capital_repair_year' as capital_repair_year,\n    ch.characteristics ->> 'total_floors_count' as floors_count,\n    ch.characteristics ->> 'occupied_floor' as occupied_floor,\n    ch.characteristics ->> 'load_bearing_walls_material' as walls_material,\n    ch.characteristics ->> 'interfloor_overlap_material' as overlap_material,\n    ch.characteristics ->> 'roofing_material' as roofing_material,\n    ch.characteristics ->> 'fire_alarm_system_availability'\n        as fire_alarm_system_availability,\n    ch.characteristics ->> 'fire_suppression_system_availability'\n        as fire_suppression_system_availability,\n    ch.characteristics ->> 'nearest_fire_station_distance_km'\n        as nearest_fire_station_distance_km,\n    ch.characteristics as object_characteristics_json,\n\n    /* адрес */\n    address.full_address,\n    address.postal_code,\n    address.region_id as address_region_id,\n    address.area as address_area,\n    address.settlement_type,\n    address.settlement,\n    address.street_type,\n    address.street,\n    address.house,\n    address.building,\n    address.block,\n    address.flat,\n    address.office,\n    address.fias_code,\n    address.longitude,\n    address.latitude,\n    address.address_dgis_id,\n\n    /* договор */\n    contract.n_contract as contract_number,\n    contract.document_status as contract_status,\n    contract.system_type as contract_source_system,\n    contract.ins_product_sbs as contract_product,\n    contract.d_sign_contract as contract_sign_date,\n    contract.d_start_contract as contract_start_date,\n    contract.d_end_contract as contract_end_date,\n    contract.prevcontract_id as previous_contract_id,\n    contract.rootcontract_id as root_contract_id,\n    previous_contract.n_contract as previous_contract_number,\n    previous_contract.d_start_contract as previous_contract_start_date,\n    previous_contract.d_end_contract as previous_contract_end_date,\n\n    /* задача и заявка */\n    task.task_type,\n    task.status as task_status,\n    task.ins_document_type,\n    task.ins_refuse,\n    task.d_create as task_create_date,\n    task.d_conclusion_ins_contract as contract_conclusion_date,\n    task.ins_product as task_product,\n    task.industry as task_industry,\n    task.subindustry as task_subindustry,\n    task.locations_count,\n    task.multi_location,\n    task.object_description as task_object_description,\n    request.business_segment,\n    request.sale_channel,\n    request.ins_product as request_product,\n\n    /* страхователь и crm */\n    policyholder.inn as policyholder_inn,\n    policyholder.company_name_short as policyholder_name,\n    policyholder.cdi_id as policyholder_cdi_id,\n    crm.segment as crm_segment,\n    crm.macroindustry as crm_macroindustry,\n    crm.industry as crm_industry,\n    crm.okved as crm_okved,\n\n    /* дата состояния строки */\n    coalesce(\n        task.d_conclusion_ins_contract::timestamp with time zone,\n        contract.d_sign_contract,\n        ch.version_start_date,\n        obj.d_create\n    ) as as_of_date\nfrom dataset_keys keys\njoin base_insurance_object obj\n    on obj.id = keys.object_id\nleft join base_insurance_object_characteristics ch\n    on ch.id = keys.characteristics_id\nleft join condition_summary conditions\n    on conditions.characteristics_id = keys.characteristics_id\nleft join bps_request_ins_task_insurance_object task_link\n    on task_link.id = keys.task_object_link_id\nleft join bps_request_ins_task task\n    on task.id = keys.task_id\nleft join bps_request_ins request\n    on request.id = keys.request_id\nleft join bps_contract contract\n    on contract.id = keys.contract_id\nleft join bps_contract previous_contract\n    on previous_contract.id = contract.prevcontract_id\nleft join bps_contractor policyholder\n    on policyholder.id = contract.contractor_id\nleft join bps_corporate_crm crm\n    on crm.id = request.corporate_crm_id\nleft join base_geo_address address\n    on address.id = obj.geo_address_id\nleft join object_link_profile profile\n    on profile.object_id = keys.object_id\norder by\n    has_contract desc,\n    as_of_date desc nulls last,\n    keys.object_id;\n"

with engine.connect() as connection:
    expanded_df = pd.read_sql_query(text(sql), connection)

print('Строк:', len(expanded_df))
print('Колонок:', len(expanded_df.columns))
display(expanded_df.head(3))

## 5. Быстрая проверка результата

В таблицах ниже нет исходных адресов, договоров или ИНН.

In [ ]:
required_columns = {
    'row_source', 'object_id', 'characteristics_id', 'contract_id',
    'elementary_obj_type', 'has_contract', 'has_address', 'has_target',
    'target_status'
}
missing_columns = sorted(required_columns - set(expanded_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных объектов',
        'Уникальных характеристик',
        'Уникальных договоров',
        'Строк с договором',
        'Строк с адресом',
        'Строк с target',
        'Строк с пустым типом объекта',
    ],
    'Значение': [
        len(expanded_df),
        expanded_df['object_id'].nunique(dropna=True),
        expanded_df['characteristics_id'].nunique(dropna=True),
        expanded_df['contract_id'].nunique(dropna=True),
        expanded_df['has_contract'].fillna(False).astype(bool).sum(),
        expanded_df['has_address'].fillna(False).astype(bool).sum(),
        expanded_df['has_target'].fillna(False).astype(bool).sum(),
        expanded_df['elementary_obj_type'].fillna('').str.strip().eq('').sum(),
    ],
})

display(profile)

In [ ]:
display(expanded_df['row_source'].fillna('empty').value_counts(dropna=False))
display(expanded_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))
display(expanded_df['target_status'].fillna('empty').value_counts(dropna=False))

## 6. Сохранение CSV

Результат сохраняется локально и не должен попадать в Git.

In [ ]:
output_path = OUTPUT_DIR / 'датасет_51_расширенный.csv'
expanded_df.to_csv(output_path, index=False, sep=';', encoding='utf-8-sig')
print('Сохранено:', output_path)

## 7. Агрегатный паспорт датасета

Отдельный скрипт повторно читает сохранённый CSV и формирует Markdown только с агрегатами.

In [ ]:
report_path = OUTPUT_DIR / 'паспорт_датасета_51.md'
subprocess.run(
    [sys.executable, str(CHECK_SCRIPT), str(output_path), '--output', str(report_path)],
    check=True,
)
print('Паспорт сохранён:', report_path)

In [ ]:
engine.dispose()
print('Подключение закрыто')